# Cuantificación de la Herencia de Diseño Compartida en una Flota de Transformadores de Potencia con PROC INBREED

## Resumen Ejecutivo

Los transformadores de subestación de un operador de red se diseñan en generaciones sucesivas de ingeniería, cada modelo nuevo derivado de dos diseños predecesores. Este notebook trata esa genealogía de ingeniería como un pedigrí y usa **PROC INBREED** para calcular los coeficientes de consanguinidad y de relación aditiva (covarianza), cuantificando cuánta herencia de diseño comparten dos activos cualesquiera.

El pedigrí se construye de modo que la estructura sea interpretable: dos de los cuatro modelos actuales de la flota descienden de un par de diseños predecesores **hermanos** y por lo tanto llevan una herencia concentrada, mientras que los otros se basan en linajes disjuntos. El procedimiento recupera esto exactamente. Los dos modelos derivados de hermanos (`G3_T1`, `G3_T3`) tienen cada uno un coeficiente de consanguinidad de **F = 0.25**; los otros dos (`G3_T2`, `G3_T4`) son **F = 0**. El coeficiente de consanguinidad promedio de la flota es **0.0417**. Leyendo la matriz de relación aditiva, el par menos relacionado de la flota actual es **`G3_T1` y `G3_T3` (relación = 0)** — no comparten ascendencia y forman el emparejamiento redundante (N-1) más sólido, lo cual importa porque `G3_T1` es en sí uno de los diseños más consanguíneos.

## Fuentes de Datos

| Conjunto de datos | Filas | Variables clave | Descripción |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Un pedigrí de ingeniería pequeño y determinista de diseños de transformadores de subestación a través de tres generaciones de diseño no superpuestas (4 plataformas fundadoras, 4 derivados de segunda generación, 4 modelos actuales de la flota). Cada diseño no fundador registra los dos diseños predecesores distintos (`Parent1`, `Parent2`) de los que se derivó. `Sex` etiqueta el rol de linaje líder del diseño (M = linaje mecánico/núcleo, F = linaje eléctrico/bobinado). El pedigrí se especifica a mano — no es aleatorio — así que la estructura de relación se conoce de antemano y puede verificarse contra la salida del procedimiento. |

# Cuantificación de la Herencia de Diseño Compartida en una Flota de Transformadores de Potencia

## Por qué a una empresa de servicios públicos le importa la "consanguinidad"

Un operador de transmisión y distribución opera cientos de transformadores de potencia de subestación. Para controlar el costo de ingeniería y el esfuerzo de calificación, los fabricantes rara vez diseñan cada transformador desde cero — cada modelo nuevo **hereda** la geometría central, la topología del bobinado, los sistemas de aislamiento y los diseños de buje de uno o dos modelos predecesores. A lo largo de varios ciclos de adquisición, esto produce una verdadera *genealogía de ingeniería*: un pedigrí de diseños.

Esa herencia compartida es un riesgo oculto de confiabilidad. Si dos transformadores que protegen la misma carga (un par redundante **N-1**) descienden del mismo diseño ancestral, es probable que un defecto de diseño latente — un modo de resonancia, un mecanismo de envejecimiento del aislamiento, una ruta de flameo del buje — esté presente en **ambas** unidades. Una única causa raíz puede entonces eliminar el par redundante simultáneamente: una *falla de modo común*.

**PROC INBREED** fue construido para cuantificar exactamente este tipo de ascendencia compartida. Aunque sus orígenes están en la cría de animales y plantas, sus matemáticas — la recursión de relación aditiva de Wright — son agnósticas al dominio: mide la fracción de herencia de diseño que comparten dos individuos a través de ancestros comunes. Mapeamos el vocabulario genético a la ingeniería de activos:

| Concepto de INBREED | Interpretación en servicios públicos |
|---|---|
| Individuo | Un diseño / modelo de transformador |
| Padre (sire/dam) | Un diseño predecesor del que se derivó |
| Generación (CLASS) | Un ciclo de adquisición / diseño |
| Coeficiente de consanguinidad *F* | Grado de herencia auto-similar dentro de un diseño |
| Covarianza / coeficiente de relación | Herencia de diseño compartida entre dos diseños |
| Par menos relacionado | Mejor emparejamiento redundante para resiliencia N-1 |

## Paso 1 — Especificar el pedigrí de diseño

Ingresamos `DesignLineage` directamente para que la estructura de relación sea inequívoca. Hay tres **generaciones de diseño no superpuestas**:

- **Generación 1** — cuatro diseños de plataforma fundadores (`G1_P1`-`G1_P4`) sin predecesores (padres en blanco).
- **Generación 2** — cuatro diseños derivados (`G2_D1`-`G2_D4`), cada uno diseñado a partir de dos plataformas de Generación 1 **distintas**. `G2_D1` y `G2_D2` son *hermanos completos* (ambos de `G1_P1` x `G1_P2`); `G2_D3` y `G2_D4` son hermanos completos (ambos de `G1_P3` x `G1_P4`).
- **Generación 3** — cuatro modelos actuales de la flota (`G3_T1`-`G3_T4`). `G3_T1` se construye a partir del par de hermanos `G2_D1` x `G2_D2`, y `G3_T3` a partir del par de hermanos `G2_D3` x `G2_D4`; estos dos diseños por lo tanto concentran herencia. `G3_T2` y `G3_T4` cruzan cada uno los dos linajes disjuntos.

Las columnas `ID`, `Parent1` y `Parent2` llevan el pedigrí; `Sex` registra qué linaje de ingeniería lideró el diseño. Un breve paso DATA de seguimiento deja en blanco el marcador `.` para que las plataformas fundadoras tengan padres vacíos, tal como lo espera PROC INBREED.

In [1]:
DATOS DesignLineage;
   LONGITUD ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   ENTRADA Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
EJECUTAR;

/* Las plataformas fundadoras no tienen predecesores: dejar en blanco el marcador */
DATOS DesignLineage;
   ESTABLECER DesignLineage;
   SI Parent1 = '.' ENTONCES Parent1 = ' ';
   SI Parent2 = '.' ENTONCES Parent2 = ' ';
EJECUTAR;

TÍTULO 'Pedigrí de Diseño de Transformadores';
PROCEDIMIENTO IMPRIMIR DATOS=DesignLineage noobs;
   VAR Generation ID Parent1 Parent2 Sex;
   ETIQUETA Generation='Generación' ID='ID de Diseño' Parent1='Predecesor 1' Parent2='Predecesor 2' Sex='Linaje';
EJECUTAR;

                                          Pedigrí de Diseño de Transformadores                                          


 Generación   ID de Diseño  Predecesor 1  Predecesor 2  Linaje
-----------  -------------  ------------  ------------  ------
          1  G1_P1                                      M
          1  G1_P2                                      M
          1  G1_P3                                      M
          1  G1_P4                                      F
          2  G2_D1          G1_P1         G1_P2         M
          2  G2_D2          G1_P1         G1_P2         F
          2  G2_D3          G1_P3         G1_P4         F
          2  G2_D4          G1_P3         G1_P4         M
          3  G3_T1          G2_D1         G2_D2         M
          3  G3_T2          G2_D1         G2_D3         F
          3  G3_T3          G2_D3         G2_D4         F
          3  G3_T4          G2_D2         G2_D4         M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Pedigrí de Diseño de Transformadores.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Paso 2 — Coeficientes de consanguinidad por generación de diseño

Ejecutamos PROC INBREED en **modo multigeneracional** nombrando `Generation` en la sentencia `CLASS`, lo que imprime el resumen del pedigrí por generación. La sentencia `VAR` suministra las tres columnas del pedigrí en el orden requerido: individuo, primer predecesor, segundo predecesor.

- La opción **IND** imprime el coeficiente de consanguinidad *F* de cada diseño — una medida de cuán concentrada está su propia herencia. Un diseño construido a partir de dos predecesores estrechamente relacionados lleva una *F* alta.
- La opción **MATRIX** imprime la matriz completa de relación aditiva para que podamos leer la herencia por pares directamente.
- La opción **AVERAGE** añade el coeficiente de consanguinidad promedio de toda la flota — un único resumen de diversidad de diseño.

Valores cercanos a 0 significan linajes independientes; *F* aumenta a medida que los dos predecesores de un diseño se vuelven más estrechamente relacionados.

In [2]:
TÍTULO 'Coeficientes de Consanguinidad por Generación de Diseño';
PROCEDIMIENTO inbreed DATOS=DesignLineage ind average MATRIX;
   VAR ID Parent1 Parent2;
   CLASE Generation;
EJECUTAR;

                                Coeficientes de Consanguinidad por Generación de Diseño                                 


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
ID                  F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to Coeficientes de Consanguinidad por Generación de Diseño.
NOTE: PROC INBREED data=DesignLineage



## Paso 3 — Coeficientes de covarianza guardados para la puntuación de riesgo posterior

Reemplazar la vista de consanguinidad con la opción **COVAR** reporta las mismas relaciones como **coeficientes de covarianza (relación aditiva)**, la forma que un equipo de gestión de activos incorporaría a una puntuación de riesgo de redundancia. La opción **OUTCOV=** captura la matriz completa como un conjunto de datos (`DesignCovar`), donde las columnas `Col1`-`Col12` guardan la relación de cada diseño con todos los demás (en el orden del pedigrí) — listas para unirse de nuevo con las asignaciones de subestación.

Imprimimos el conjunto de datos de salida para que la matriz guardada sea visible junto al listado.

In [3]:
TÍTULO 'Coeficientes de Covarianza (Relación)';
PROCEDIMIENTO inbreed DATOS=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   VAR ID Parent1 Parent2;
   CLASE Generation;
EJECUTAR;

TÍTULO 'Matriz de Relación OUTCOV= Guardada para Puntuación de Riesgo';
PROCEDIMIENTO IMPRIMIR DATOS=DesignCovar noobs;
EJECUTAR;

                                         Coeficientes de Covarianza (Relación)                                          


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
ID                     G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to Coeficientes de Covarianza (Relación).
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to Matriz de Relación OUTCOV= Guardada para Puntuación de Riesgo.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Paso 4 — Emparejamientos menos relacionados para instalaciones redundantes (N-1)

La matriz `DesignCovar` guardada es exactamente lo que necesita un estudio de redundancia. La relación del diseño *i* con el diseño *j* vive en la fila *i*, columna `Col`*j* (las columnas están en el orden del pedigrí, así que `Col9`-`Col12` corresponden a los cuatro modelos actuales de la flota `G3_T1`-`G3_T4`).

Leemos las cuatro filas de la flota actual de `DesignCovar`, formamos cada par no ordenado de la flota actual, adjuntamos su coeficiente de relación y ordenamos de menor a mayor relación. El objetivo es elegir pares redundantes cuya herencia compartida sea **más baja** — esos minimizan la probabilidad de que un defecto de diseño heredado deshabilite ambas mitades de una posición protegida N-1.

In [4]:
/* Extraer las cuatro filas de la flota actual; Col9..Col12 son las relaciones
   con G3_T1..G3_T4 (orden de columnas del pedigrí). Formar cada par no ordenado. */
DATOS Pairs;
   ESTABLECER DesignCovar;
   DONDE ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   LONGITUD DesignA $12 DesignB $12;
   ARREGLO g3{4} Col9 Col10 Col11 Col12;
   ARREGLO nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   HACER j = 1 HASTA 4;
      DesignB = nm{j};
      SI DesignA < DesignB ENTONCES HACER;
         Relationship = g3{j};
         SALIDA;
      END;
   END;
   MANTENER DesignA DesignB Relationship;
EJECUTAR;

PROCEDIMIENTO ORDENAR DATOS=Pairs; POR Relationship; EJECUTAR;

TÍTULO 'Emparejamientos Redundantes (N-1) Candidatos, del Menos Relacionado al Más';
PROCEDIMIENTO IMPRIMIR DATOS=Pairs noobs;
   VAR DesignA DesignB Relationship;
   ETIQUETA DesignA='Diseño A' DesignB='Diseño B' Relationship='Relación';
EJECUTAR;
TÍTULO;

                       Emparejamientos Redundantes (N-1) Candidatos, del Menos Relacionado al Más                       


 Diseño A   Diseño B   Relación
---------  ---------  ---------
G3_T1      G3_T3              0
G3_T2      G3_T4           0.25
G3_T1      G3_T2          0.375
G3_T1      G3_T4          0.375
G3_T2      G3_T3          0.375
G3_T3      G3_T4          0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Emparejamientos Redundantes (N-1) Candidatos, del Menos Relacionado al Más.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Interpretación de los resultados

**Coeficientes de consanguinidad (Paso 2).** Todas las plataformas fundadoras de Generación 1 y todos los derivados de Generación 2 muestran *F* = 0 — por construcción ninguno tiene dos predecesores relacionados. Dos modelos de Generación 3 aparecen con *F* = 0.25: `G3_T1` (construido a partir del par de hermanos `G2_D1` x `G2_D2`) y `G3_T3` (a partir del par de hermanos `G2_D3` x `G2_D4`). Sus predecesores se remontan al *mismo* par de plataformas fundadoras, así que su herencia está concentrada. Desde el punto de vista de la confiabilidad, estos son los diseños más expuestos a un único defecto heredado, y merecen pruebas de calificación independientes adicionales. `G3_T2` y `G3_T4`, que cruzan los dos linajes disjuntos, son *F* = 0.

**Matriz de relación (Paso 3).** Las entradas fuera de la diagonal cuantifican la herencia compartida por pares. Los dos pares hermanos de Generación 2 muestran cada uno una relación de 0.50 entre sí (`G2_D1`-`G2_D2` y `G2_D3`-`G2_D4`), mientras que los derivados de linajes opuestos se ubican en 0.00. Los diseños consanguíneos de la flota actual llevan autorrelaciones de 1.25 (= 1 + *F*), visibles en la diagonal. El conjunto de datos `DesignCovar` pone la matriz completa a disposición para unirla con las asignaciones de subestación.

**Emparejamientos menos relacionados (Paso 4).** Clasificar cada par de la flota actual por relación sitúa a **`G3_T1` y `G3_T3` primero con una relación de 0.00** — descienden de plataformas ancestrales completamente disjuntas y no comparten herencia de diseño. Este es el emparejamiento redundante más sólido, y es especialmente valioso porque `G3_T1` es en sí uno de los dos diseños más consanguíneos: emparejarlo con una contraparte completamente no relacionada cubre su herencia concentrada. El siguiente mejor par es `G3_T2` y `G3_T4` con 0.25; los pares restantes se ubican en 0.375. El coeficiente de consanguinidad promedio de la flota de **0.0417** (impreso por la opción AVERAGE en el Paso 2) resume la diversidad general de diseño. Las adquisiciones deberían preferir los pares de menor relación para las posiciones N-1 y, con el tiempo, ampliar la base ancestral — el equivalente en ingeniería de activos de evitar un cuello de botella genético.

**Advertencia.** Estos son datos sintéticos ilustrativos; un estudio de producción construiría el pedigrí a partir de los registros reales de revisión de diseño del fabricante y validaría las puntuaciones de relación contra eventos históricos de falla de modo común antes de impulsar decisiones de adquisición.